## Assignment: Personal Chef

Objective:
A personal chef assistant that takes ingredients and creates recipes with them and can answer any follow-up question. Input can be also an image with the leftover ingredients.

Criteria:
- agent takes text/image data as input
- agent uses web search for recipes
- agent uses memory/state

In [44]:
from io import BytesIO

from dotenv import load_dotenv
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from ipywidgets import FileUpload
from IPython.display import display
import base64
from pprint import pprint
from PIL import Image

load_dotenv()

True

In [23]:
# tools
tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    return tavily_client.search(query)

In [53]:
# agent
system_prompt = """\
You are a personal chef assistant and suggest recipes based \
on the ingredients available to the user. The user either \
mentions available ingredients in the user message or uploads \
an image showing available ingrdients.\

If an image is uploaded analyze the ingredients that are shown.

# Important
To suggest recipes you MUST use the available tool web_search and ask for recipes with the user's ingredients. \
Afterwards, summarize the web results and present a listed view \
to the user with possible dishes and a short description about each dish and the link to the recipe, but don't include the full recipe in your response yet.
"""

chef_agent = create_agent(
    model="gpt-5-nano",
    system_prompt=system_prompt,
    tools=[web_search],
    checkpointer=InMemorySaver(),
)

print(system_prompt)

You are a personal chef assistant and suggest recipes based on the ingredients available to the user. The user either mentions available ingredients in the user message or uploads an image showing available ingrdients.
If an image is uploaded analyze the ingredients that are shown.

# Important
To suggest recipes you MUST use the available tool web_search and ask for recipes with the user's ingredients. Afterwards, summarize the web results and present a listed view to the user with possible dishes and a short description about each dish and the link to the recipe, but don't include the full recipe in your response yet.



In [54]:
def upload_file() -> FileUpload:
    uploader = FileUpload(accept=".png,.jpg,.jpeg", multiple=False)
    display(uploader)
    return uploader


uploader = upload_file()

FileUpload(value=(), accept='.png,.jpg,.jpeg', description='Upload')

In [55]:
if uploader.value:
    img_bytes = bytes(uploader.value[0]["content"])
    with Image.open(BytesIO(img_bytes)) as img:
        mime_type = Image.MIME[img.format]

    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    question = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "This is what I have left, what can I make with it?",
            },
            {"type": "image", "base64": img_b64, "mime_type": mime_type},
        ]
    )
else:
    question = HumanMessage(
        content=[
            {
                "type": "text",
                "text": "I have ground beef and pasta left. What can I cook? Please give me 3 very different recipes.",
            }
        ]
    )

question.content[0]["text"]

'I have ground beef and pasta left. What can I cook? Please give me 3 very different recipes.'

In [56]:
config = {"configurable": {"thread_id": "3"}}

response = chef_agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'I have ground beef and pasta left. What can I cook? Please give me 3 very different recipes.'}], additional_kwargs={}, response_metadata={}, id='e33be50e-5471-4e4e-a451-1d5759e117bb'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1243, 'prompt_tokens': 264, 'total_tokens': 1507, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZOVWPrxzdODSAgmw8z3LWzhHBCTK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a141be-e9d2-7de3-ac2c-1bb38088a559-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'classic spa

In [58]:
print(response["messages"][-1].content)

Nice! Here are three very different ways to use ground beef and pasta. Each link goes to a full recipe, but I’m not including the whole recipe here.

1) One-Pot Creamy Tomato Beef Pasta (RecipeTin Eats)
- What it is: A fast, weeknight-friendly dish where everything simmers in one pot in a creamy tomato sauce.
- Why it’s different: Creamy, tomato-forward sauce with minimal cleanup.
- Link: https://www.recipetineats.com/one-pot-creamy-tomato-beef-pasta

2) Pasta al Forno (Easy Baked Pasta) with Meat Sauce
- What it is: A comforting baked pasta dish layered with meat sauce and cheese, then baked until bubbling.
- Why it’s different: Baked, gooey-cheesy casserole experience; great for leftovers.
- Link: https://www.savoringitaly.com/pasta-al-forno

3) Ragù alla Bolognese (Authentic Ragù alla Bolognese)
- What it is: A traditional slow-simmered meat sauce (usually with soffritto, wine, and a touch of milk) served with pasta like tagliatelle or pappardelle.
- Why it’s different: Classic, dee

In [59]:
question = HumanMessage(
    content="The second dish sounds awesome. How can I improve it and make it rich in diverse flavors?"
)

In [60]:
response = chef_agent.invoke(
    {"messages": [question]},
    config,
)

pprint(response)
print(response["messages"][-1].content)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'I have ground beef and pasta left. What can I cook? Please give me 3 very different recipes.'}], additional_kwargs={}, response_metadata={}, id='e33be50e-5471-4e4e-a451-1d5759e117bb'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1243, 'prompt_tokens': 264, 'total_tokens': 1507, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EZOVWPrxzdODSAgmw8z3LWzhHBCTK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a141be-e9d2-7de3-ac2c-1bb38088a559-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'classic spa